In [ ]:
from google.colab import drive
drive.mount('/content/drive')
base_path = "/content/drive/MyDrive/video_detection"

convlstm_path = base_path + "/convlstm_motion_predictor.keras"
yolo_path = base_path + "/Yolo_model/best.pt"
video_path = base_path + "/video/Test001.mp4"

In [ ]:
!pip install ultralytics -q

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import load_model
from ultralytics import YOLO

convlstm_model = load_model(convlstm_path)
yolo_model = YOLO(yolo_path)

print("ConvLSTM loaded")
print("YOLO loaded")
print("Video exists:", os.path.exists(video_path))

In [ ]:
FRAME_SIZE = (128, 128)
SEQUENCE_LENGTH = 10

cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

original_frames = []
processed_frames = []

while True:
    ret, frame = cap.read()
    if not ret:
        break

    original_frames.append(frame)

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, FRAME_SIZE)
    gray = gray.astype(np.float32) / 255.0

    processed_frames.append(gray)

cap.release()

original_frames = np.array(original_frames)
processed_frames = np.array(processed_frames, dtype=np.float32)

print("FPS:", fps)
print("Original size:", width, height)
print("Total frames:", len(original_frames))
print("Processed frames:", processed_frames.shape)

In [ ]:
motion_maps = []

for i in range(1, len(processed_frames)):
    motion = np.abs(processed_frames[i] - processed_frames[i - 1])
    motion_maps.append(motion[..., np.newaxis])

motion_maps = np.array(motion_maps, dtype=np.float32)

print("Motion maps:", motion_maps.shape)

In [ ]:
X_video = []
target_frame_indices = []

for i in range(len(motion_maps) - SEQUENCE_LENGTH + 1):
    seq = motion_maps[i : i + SEQUENCE_LENGTH]

    X_video.append(seq[:SEQUENCE_LENGTH - 1])

    # target motion map is seq[9], which corresponds approx to frame i + 10
    target_frame_indices.append(i + SEQUENCE_LENGTH)

X_video = np.array(X_video, dtype=np.float32)
target_frame_indices = np.array(target_frame_indices)

print("X_video:", X_video.shape)
print("Target frame indices:", target_frame_indices.shape)

In [ ]:
pred_video = convlstm_model.predict(X_video, batch_size=4, verbose=1)

y_video = motion_maps[SEQUENCE_LENGTH - 1:]

video_errors = np.mean((pred_video - y_video) ** 2, axis=(1, 2, 3))


print("Video errors:", video_errors.shape)
print("Min:", np.min(video_errors))
print("Max:", np.max(video_errors))
print("Mean:", np.mean(video_errors))

In [ ]:
clip_threshold = 0.0009591725
frame_threshold = 0.00053535208   # this threshold is changable 
anomaly_mask = video_errors > frame_threshold
anomaly_frames = target_frame_indices[anomaly_mask]

print("Number of anomaly frames:", len(anomaly_frames))
print("Anomaly frames:", anomaly_frames)

In [ ]:
def get_anomaly_segments(anomaly_frames, max_gap=100, min_length=3):
    if len(anomaly_frames) == 0:
        return []

    anomaly_frames = sorted(anomaly_frames)

    segments = []
    start = anomaly_frames[0]
    prev = anomaly_frames[0]

    for frame in anomaly_frames[1:]:
        if frame - prev <= max_gap:
            prev = frame
        else:
            if prev - start + 1 >= min_length:
                segments.append((start, prev))
            start = frame
            prev = frame

    if prev - start + 1 >= min_length:
        segments.append((start, prev))

    return segments


segments = get_anomaly_segments(
    anomaly_frames,
    max_gap=100,
    min_length=3
)

print("Anomaly segments:", segments)

In [ ]:
plt.figure(figsize=(16, 5))
plt.plot(target_frame_indices, video_errors, label="Frame error")
plt.axhline(frame_threshold, linestyle="--", label="Frame threshold")
plt.axhline(clip_threshold, linestyle="--", label="Clip threshold")

for start, end in segments:
    plt.axvspan(start, end, alpha=0.3)

plt.xlabel("Frame Index")
plt.ylabel("Prediction Error")
plt.title("Frame-Level Anomaly Timeline")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
output_path = base_path + "/final_videos//test01.mp4"

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

anomaly_frame_set = set()

for start, end in segments:
    for frame_idx in range(start, end + 1):
        anomaly_frame_set.add(frame_idx)

for idx, frame in enumerate(original_frames):
    annotated_frame = frame.copy()

    if idx in anomaly_frame_set:
        results = yolo_model(annotated_frame, conf=0.50, verbose=False)

        annotated_frame = results[0].plot()

        cv2.putText(
            annotated_frame,
            "ANOMALY DETECTED",
            (10, 25),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 0, 255),
            2
        )

    out.write(annotated_frame)

out.release()

print("Saved output video to:", output_path)